# Anime Face Decals (FLUX on T4)

Generates 12 anime face decals on transparent background — eyes/blush/mouth/nose only, no face outline, no chin, no hair.

**How to run:**
1. Runtime > Change runtime type > **T4 GPU**
2. Run all cells top-to-bottom (Runtime > Run all)
3. A zip with 12 transparent PNGs auto-downloads to your machine when done

Total time: ~5 minutes (most of it loading FLUX the first time).

## 1. Check GPU

In [ ]:
!nvidia-smi | head -n 15

## 2. Authenticate with HuggingFace

Paste your HF token (https://huggingface.co/settings/tokens, read scope is fine).

In [ ]:
from huggingface_hub import login
import getpass
login(token=getpass.getpass('HF token: '))

## 3. Install + load FLUX-schnell

Downloads ~24 GB of FLUX weights the first time (3-4 min). Cached for next session.

In [ ]:
!pip install -q diffusers accelerate sentencepiece protobuf

import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.bfloat16,
)
pipe.enable_sequential_cpu_offload()  # fits on T4
print('FLUX ready')

## 4. Generate 12 face decals

Each renders on a magenta background. We keep magenta-keying cleaner than white-keying because white pixels inside the features (eye highlights, teeth, tears) survive.

In [ ]:
from pathlib import Path
import torch

out_dir = Path('/content/face_decals')
out_dir.mkdir(exist_ok=True)

SUFFIX = (
    'ONLY isolated facial features as separate floating elements, '
    'no face outline, no chin, no jaw, no cheeks outline, no ears, no head, '
    'no hair, no neck, no body, no skin shape, '
    'features drawn directly on solid magenta background pure #FF00FF, '
    'the features are NOT inside any face shape, '
    'vector sticker art, flat 2D anime style, clean black line art, '
    'high contrast, centered, eye highlights are white'
)

variants = [
    ('01_sparkly',    'huge sparkling pink anime eyes with star-shaped white highlights, small gentle smile curve mouth, two pink hatched blush patches, tiny nose dot'),
    ('02_shy',        'soft round blue anime eyes with white highlights looking down, small bashful curved mouth, two soft pink blush ovals, tiny nose dot'),
    ('03_smug',       'narrow green cat-like anime eyes with white highlights, smirk mouth with one tiny white fang, small blush, tiny nose'),
    ('04_dreamy',     'closed sleepy anime eyes drawn as long curved black eyelashes, peaceful tiny smile curve, light pink blush, tiny nose'),
    ('05_starry',     'huge anime eyes with star-shaped white pupils, wide open round excited smile with white teeth, hatched pink blush, tiny nose'),
    ('06_tsundere',   'red angry anime eyes with sharp black eyebrows above them, small pouting mouth, deep pink blush, tiny nose'),
    ('07_glasses',    'round black glasses outline with brown anime eyes inside, gentle smile curve, light blush, tiny nose'),
    ('08_lovestruck', 'anime eyes with pink heart-shaped pupils, blissful curved smile, big pink blush patches, tiny nose'),
    ('09_determined', 'sharp amber anime eyes with intense angled black eyebrows above, firm closed neutral mouth, tiny nose'),
    ('10_peaceful',   'closed curved black eyelash arcs eyes, gentle small smile curve, soft pink blush, tiny nose'),
    ('11_playful',    'one open blue anime eye with white highlight and one winked closed eye, tongue sticking out playfully with white teeth, blush, tiny nose'),
    ('12_teary',      'round watery anime eyes with white highlights and shiny white teardrops below, trembling pouty small mouth, light blush, tiny nose'),
]

for slug, mod in variants:
    prompt = f'{mod}. {SUFFIX}'
    gen = torch.Generator(device='cpu').manual_seed(hash(slug+'v3') & 0x7FFFFFFF)
    img = pipe(
        prompt=prompt,
        width=1024,
        height=1024,
        num_inference_steps=4,
        guidance_scale=0.0,
        generator=gen,
    ).images[0]
    img.save(out_dir / f'{slug}.png')
    print(f'  {slug}.png')

print('DONE')

## 5. Color-key magenta to transparent + download

Zips all 12 transparent PNGs and auto-downloads to your computer.

In [ ]:
import numpy as np
from PIL import Image
import shutil
from pathlib import Path
from google.colab import files

src = Path('/content/face_decals')
out = Path('/content/face_decals_transparent')
out.mkdir(exist_ok=True)

for png in sorted(src.glob('*.png')):
    im = Image.open(png).convert('RGBA')
    a = np.array(im).astype(np.float32)
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    magenta = (r > 200) & (g < 80) & (b > 200)
    mag_strength = np.clip(((r - 150) / 100.0) * ((255 - g) / 200.0) * ((b - 150) / 100.0), 0, 1)
    alpha = (1 - mag_strength) * 255
    alpha[magenta] = 0
    a[..., 3] = alpha
    a[..., :3] = np.where(alpha[..., None] > 0, a[..., :3], 0)
    Image.fromarray(a.astype(np.uint8), 'RGBA').save(out / png.name)

shutil.make_archive('/content/face_decals_transparent', 'zip', out)
print('zipped, downloading...')
files.download('/content/face_decals_transparent.zip')